# SolarTriage — model v2 pe Kaggle (augmentări + label smoothing + seed)

Rulează 3 configurații, alege pe **val** varianta cu cel mai bun macro-F1 după boost (TTA + bias),
apoi exportă ONNX și recalibrează. Baseline de bătut: **val 0,719** (modelul v1.1).

Necesită: Accelerator **GPU T4**, Internet **On**, input-ul `solartriage-ml` (folderul `ml/` din repo).


In [ ]:
import importlib.util, json, shutil, subprocess, sys
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Settings → Accelerator → GPU T4"
print(torch.__version__, torch.cuda.get_device_name(0))

src = next(p.parent for p in Path("/kaggle/input").rglob("train.py")
           if (p.parent / "splits").is_dir())
REPO = Path("/tmp/repo")
shutil.copytree(src, REPO / "ml", dirs_exist_ok=True)
print("cod din", src)

def run(*args):
    subprocess.run([sys.executable, *args], cwd=REPO, check=True)


In [ ]:
# Imaginile Raptor (15,5 MB, SHA-256 verificat); split-urile vin din repo.
run("-c", "import sys; sys.path.insert(0, 'ml'); import data; data.download(); data.extract()")


In [ ]:
AUG = ["--label-smoothing", "0.05", "--glare-prob", "0.10",
       "--blur-prob", "0.15", "--brightness", "0.30"]
BASE = ["--device", "cuda", "--epochs", "30"]
RUNS = {
    "v2_ls": ["--seed", "42", "--label-smoothing", "0.05"],
    "v2_aug": ["--seed", "42", *AUG],
    "v2_aug_seed123": ["--seed", "123", *AUG],
}


In [ ]:
for name, flags in RUNS.items():
    out = f"/kaggle/working/{name}"
    print(f"=== {name}: {flags}")
    run("ml/train.py", *BASE, *flags, "--out", out)
    run("ml/boost.py", "--ckpt", f"{out}/best.pt")  # doar cifrele pe val, fără export
    shutil.copy(REPO / "reports" / "boost_val.json", f"{out}/boost_val.json")


In [ ]:
rows = []
for name in RUNS:
    r = json.load(open(f"/kaggle/working/{name}/val_report.json"))["best"]
    b = json.load(open(f"/kaggle/working/{name}/boost_val.json"))
    rows.append((name, r["macro_f1"], b["variants"][b["best"]]["macro_f1"], b["best"]))
rows.sort(key=lambda t: -t[2])

print(f"{'run':<18}{'raw val':>9}{'boost val':>11}  variant")
for n, raw, boost, var in rows:
    print(f"{n:<18}{raw:>9.4f}{boost:>11.4f}  {var}")

BASELINE = 0.7194
best_name, _, best_f1, _ = rows[0]
EXPORT_OK = best_f1 > BASELINE
print(f"baseline v1.1: {BASELINE:.3f}; cel mai bun: {best_name} {best_f1:.4f}")
print("export:", "DA" if EXPORT_OK else "NU - nu bate baseline-ul")


In [ ]:
assert EXPORT_OK, "Nu exporta: niciun run nu bate baseline-ul pe val."

for pkg in ("onnx", "onnxruntime"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

ckpt = f"/kaggle/working/{best_name}/best.pt"
run("ml/boost.py", "--ckpt", ckpt, "--export")
art = REPO / "backend" / "artifacts" / "model.onnx"
if not art.exists():  # TTA+bias n-a bătut varianta simplă pe val
    run("ml/export_onnx.py", "--ckpt", ckpt,
        "--out-dir", str(REPO / "backend" / "artifacts"))
run("ml/calibrate.py")  # T și praguri noi, pentru modelul nou

shutil.copy(ckpt, "/kaggle/working/best.pt")
shutil.copy(f"/kaggle/working/{best_name}/val_report.json", "/kaggle/working/val_report.json")
shutil.copytree(REPO / "backend" / "artifacts", "/kaggle/working/artifacts",
                dirs_exist_ok=True)
shutil.copy(REPO / "reports" / "boost_val.json", "/kaggle/working/boost_val.json")


In [ ]:
th = json.load(open("/kaggle/working/artifacts/thresholds.json"))
print("praguri:", th["min_conf"])
print("acoperire val:", round(th["val"]["coverage"], 3),
      "precizie:", round(th["val"]["precision_auto"], 3))
!ls -la /kaggle/working /kaggle/working/artifacts
